# 🎨 Antigravity 4K WebGL Live Studio & GPU Batch Renderer (Full Interactive UI in Colab)
Studio Pembuat Video Loop 4K Motion Graphics Lengkap dengan **UI Visual Interaktif Langsung di Google Colab**!

### ✨ Kemudahan:
1. **Buka UI Langsung di Colab** tanpa perlu copy-paste JSON manual.
2. Atur slider warna, shader Paper & ShaderGradient, dan preview real-time langsung di dalam tab browser.
3. Tekan tombol **Export / Render 4K** langsung dari UI dan video 4K otomatis ter-download ke PC/Laptop Anda!

## ⚙️ Step 1: Install Environment & Jalankan Web Studio UI di Colab

In [ ]:
import os, subprocess, shutil, time

os.chdir('/content')

print("⏳ [1/3] Menyiapkan environment GPU & Dependencies...")
!apt-get install -y ffmpeg libgbm-dev libnss3 libasound2 zip > /dev/null 2>&1

print("⏳ [2/3] Mengunduh Studio UI terbaru dari GitHub...")
if os.path.exists('/content/shadergradientpaper'):
    shutil.rmtree('/content/shadergradientpaper', ignore_errors=True)

!git clone https://github.com/consistmaker/shadergradientpaper.git /content/shadergradientpaper

print("⏳ [3/3] Membangun Aplikasi WebGL Studio...")
%cd /content/shadergradientpaper
!npm install --legacy-peer-deps > /dev/null 2>&1
!npm run build
%cd /content

print("\n✅ STUDIO BERHASIL DIBANGUN!")

## 🌐 Step 2: Buka Live UI Studio Langsung di Google Colab (2 Pilihan Akses)
Jalankan cell di bawah ini. Anda bisa membuka UI langsung **di dalam panel Colab (IFrame)** atau lewat **Link Publik Bebas Hambatan**!

In [ ]:
import http.server, socketserver, threading, os, time
from IPython.display import display, HTML
from google.colab.output import eval_js

PORT = 8080
os.chdir('/content/shadergradientpaper/dist')

class QuietHandler(http.server.SimpleHTTPRequestHandler):
    def log_message(self, format, *args):
        pass

# Jalankan Local Server di background thread
try:
    socketserver.TCPServer.allow_reuse_address = True
    httpd = socketserver.TCPServer(("0.0.0.0", PORT), QuietHandler)
    threading.Thread(target=httpd.serve_forever, daemon=True).start()
except Exception as e:
    pass

time.sleep(1)
colab_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")

print(f"""
========================================================================
🚀 LIVE WEBGL STUDIO UI BERHASIL DIAKTIFKAN!

👉 Link Akses Tab Baru: {colab_url}
========================================================================
""")

# Tampilkan UI Studio langsung di dalam notebook Colab!
display(HTML(f'''
    <div style="border: 2px solid #6366f1; border-radius: 12px; overflow: hidden; margin-top: 10px;">
        <div style="background: #0f172a; color: #fff; padding: 10px 16px; font-weight: bold; font-size: 14px; display: flex; justify-content: space-between; align-items: center;">
            <span>🎨 Live Studio Window (Colab Embedded)</span>
            <a href="{colab_url}" target="_blank" style="color: #38bdf8; text-decoration: none; font-size: 12px; background: rgba(56, 189, 248, 0.1); padding: 4px 8px; border-radius: 4px;">↗ Buka di Tab Baru</a>
        </div>
        <iframe src="{colab_url}" width="100%" height="750px" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture"></iframe>
    </div>
'''))

## 🎬 Step 3: Fast GPU Batch Render Engine (Otomatis Render & Download ke PC)

In [ ]:
import subprocess, time, os, glob
from google.colab import files

output_dir = '/content/output_4k_videos'
os.makedirs(output_dir, exist_ok=True)
downloaded_files = set()

print("🚀 Starting 4K GPU Render Engine...")

process = subprocess.Popen(
    ['node', '/content/shadergradientpaper/headless_renderer.cjs'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1
)

while True:
    line = process.stdout.readline()
    if line:
        print(line, end='')
        if '✅ Success 4K Render:' in line:
            time.sleep(1)
            current_videos = glob.glob(f"{output_dir}/*.mp4")
            for v_path in current_videos:
                if v_path not in downloaded_files and os.path.exists(v_path):
                    file_size = (os.path.getsize(v_path) / (1024 * 1024))
                    print(f"   📥 [AUTO-DOWNLOAD] Mengunduh: {os.path.basename(v_path)} ({file_size:.2f} MB)...")
                    try:
                        files.download(v_path)
                        downloaded_files.add(v_path)
                    except Exception as e:
                        pass

    if process.poll() is not None:
        for remaining in process.stdout.readlines():
            print(remaining, end='')
        break

print(f"\n🎉 SELESAI! Video 4K telah siap.")